In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg_react.yaml
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000525.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000595.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/_annotations.auto.coco.json
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000507.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000563.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000538.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000547.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000532.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000534.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000504.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/val/group_5_000508.jpg
/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg

In [2]:
!ls -R /kaggle/input/ | head -40

/kaggle/input/:
datasets

/kaggle/input/datasets:
shah9212

/kaggle/input/datasets/shah9212:
spatial-sgg

/kaggle/input/datasets/shah9212/spatial-sgg:
spatial_sgg
spatial_sgg_react.yaml
spatial_sgg_yolo

/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg:
test
train
val

/kaggle/input/datasets/shah9212/spatial-sgg/spatial_sgg/test:
_annotations.auto.coco.json
_annotations.human.coco.json
group_6_000600.jpg
group_6_000601.jpg
group_6_000602.jpg
group_6_000603.jpg
group_6_000604.jpg
group_6_000605.jpg
group_6_000606.jpg
group_6_000607.jpg
group_6_000608.jpg
group_6_000609.jpg
group_6_000610.jpg
group_6_000611.jpg
group_6_000612.jpg
group_6_000613.jpg
group_6_000614.jpg
group_6_000615.jpg
group_6_000616.jpg
group_6_000617.jpg
ls: write error: Broken pipe


In [3]:
%%bash
cd /kaggle/working && rm -rf SGG-Benchmark
git clone -q https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark && pip install -e . -q
pip install -q ultralytics hydra-core omegaconf
echo "INSTALL DONE"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 9.9 MB/s eta 0:00:00
INSTALL DONE


In [4]:
%%bash
set -e
BASE=/kaggle/working/SGG-Benchmark
INPUT=$(dirname $(find /kaggle/input -name spatial_sgg_react.yaml | head -1))
echo "found data at: $INPUT"
mkdir -p $BASE/datasets $BASE/configs/hydra/Spatial
cp -r $INPUT/spatial_sgg $BASE/datasets/
cp -r $INPUT/spatial_sgg_yolo $BASE/datasets/
cp $INPUT/spatial_sgg_react.yaml $BASE/configs/hydra/Spatial/react.yaml
echo "DATA COPIED"; ls $BASE/datasets

found data at: /kaggle/input/datasets/shah9212/spatial-sgg
DATA COPIED
psg
spatial_sgg
spatial_sgg_yolo
vg


In [5]:
import json, glob, os
os.chdir("/kaggle/working/SGG-Benchmark")
for p in sorted(glob.glob("datasets/spatial_sgg/*/_annotations.human.coco.json") +
                glob.glob("datasets/spatial_sgg/*/_annotations.auto.coco.json")):
    d = json.load(open(p))
    if any(c["name"] == "__background__" for c in d["categories"]):
        continue
    for c in d["categories"]:      c["id"] += 1
    for c in d["rel_categories"]:  c["id"] += 1
    for a in d["annotations"]:     a["category_id"]  += 1
    for r in d["rel_annotations"]: r["predicate_id"] += 1
    d["categories"].insert(0, {"id": 0, "name": "__background__", "supercategory": "none"})
    d["rel_categories"].insert(0, {"id": 0, "name": "__no_relation__"})
    json.dump(d, open(p, "w"))
d = json.load(open("datasets/spatial_sgg/test/_annotations.human.coco.json"))
assert d["categories"][0]["name"] == "__background__" and len(d["categories"]) == 7
assert d["rel_categories"][0]["name"] == "__no_relation__" and len(d["rel_categories"]) == 8
print("PATCH OK — 7 object classes (bg+6), 8 relation classes (norel+7)")

PATCH OK — 7 object classes (bg+6), 8 relation classes (norel+7)


In [6]:
import os, yaml, shutil, glob
os.chdir("/kaggle/working/SGG-Benchmark")
yp = "datasets/spatial_sgg_yolo/data.yaml"
d = yaml.safe_load(open(yp)); d["path"] = os.path.abspath("datasets/spatial_sgg_yolo")
yaml.safe_dump(d, open(yp, "w"))
from ultralytics import YOLO
YOLO("yolov8m.pt").train(data=yp, epochs=60, imgsz=640, batch=16,
                         project="det", name="yolov8m_spatial", verbose=False)
os.makedirs("checkpoints/BACKBONES", exist_ok=True)
src = max(glob.glob("runs/detect/det/yolov8m_spatial*/weights/best.pt"), key=os.path.getmtime)
shutil.copy(src, "checkpoints/BACKBONES/yolov8m_spatial.pt")
print("DETECTOR DONE ->", src)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/spatial_sgg_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, h

In [7]:
import subprocess, os, shutil, re
os.chdir("/kaggle/working/SGG-Benchmark")
for s in ["train","val","test"]:
    shutil.copy(f"datasets/spatial_sgg/{s}/_annotations.human.coco.json",
                f"datasets/spatial_sgg/{s}/_annotations.coco.json")
os.system("rm -rf checkpoints/spatial/smoke")
cmd = ("python tools/relation_train_net_hydra.py --config-path ../configs/hydra/Spatial "
       "--config-name react --task sgdet --save-best "
       "solver.max_epoch=1 output_dir=./checkpoints/spatial/smoke")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
out = r.stdout + "\n" + r.stderr
print(out[-2500:])
mrs = [float(x) for x in re.findall(r"Result for mR:\s*([\d.]+)", out)]
print("\n>>> smoke mR:", mrs)
assert mrs and max(mrs) > 0, "SMOKE FAILED (mR still zero) — stop and tell Claude"
print(">>> SMOKE PASSED — full arms will run")

200d.txt: 100%|██████████| 400000/400000 [00:20<00:00, 19718.88it/s]
/kaggle/working/SGG-Benchmark/sgg_benchmark/engine/trainer.py:334: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"{_ALIASES.get(k, k)}={float(v):.3f}" for k, v in loss_dict_reduced.items()

100%|██████████| 100/100 [00:04<00:00, 24.59it/s]

SGG Eval: 100%|██████████| 100/100 [00:00<00:00, 280.10it/s]


>>> smoke mR: [0.0296]
>>> SMOKE PASSED — full arms will run


In [8]:
%%bash
cd /kaggle/working/SGG-Benchmark
for s in train val test; do
  cp datasets/spatial_sgg/$s/_annotations.human.coco.json datasets/spatial_sgg/$s/_annotations.coco.json
done
rm -rf checkpoints/spatial/react_human
python tools/relation_train_net_hydra.py --config-path ../configs/hydra/Spatial \
  --config-name react --task sgdet --save-best \
  output_dir=./checkpoints/spatial/react_human
echo "ARM HUMAN DONE"

Using Hydra + OmegaConf config mode...
Config path: ../configs/hydra/Spatial
Config name: react
Mode: TRAINING
Loading config from: /kaggle/working/SGG-Benchmark/configs/hydra/Spatial/react.yaml
2026-07-12 19:18:16,966 sgg_benchmark INFO: Using 1 GPUs
2026-07-12 19:18:16,967 sgg_benchmark INFO: Task mode: sgdet
2026-07-12 19:18:16,967 sgg_benchmark INFO: Loading training dataset to extract class information...
2026-07-12 19:18:17,010 sgg_benchmark INFO: Extracted from dataset: num_obj_classes=7, num_rel_classes=8
2026-07-12 19:18:17,011 sgg_benchmark INFO: Saving config to: ./checkpoints/spatial/react_human/config.yml
2026-07-12 19:18:17,039 sgg_benchmark INFO: Saving Hydra config to: ./checkpoints/spatial/react_human/hydra_config.yaml
2026-07-12 19:18:17,039 sgg_benchmark INFO: Building model...
Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  u

/kaggle/working/SGG-Benchmark/sgg_benchmark/engine/trainer.py:334: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"{_ALIASES.get(k, k)}={float(v):.3f}" for k, v in loss_dict_reduced.items()
SGG Eval: 100%|██████████| 100/100 [00:00<00:00, 220.67it/s]


In [9]:
%%bash
cd /kaggle/working/SGG-Benchmark
for s in train val; do
  cp datasets/spatial_sgg/$s/_annotations.auto.coco.json datasets/spatial_sgg/$s/_annotations.coco.json
done
cp datasets/spatial_sgg/test/_annotations.human.coco.json datasets/spatial_sgg/test/_annotations.coco.json
rm -rf checkpoints/spatial/react_auto
python tools/relation_train_net_hydra.py --config-path ../configs/hydra/Spatial \
  --config-name react --task sgdet --save-best \
  output_dir=./checkpoints/spatial/react_auto
echo "ARM AUTO DONE"

Using Hydra + OmegaConf config mode...
Config path: ../configs/hydra/Spatial
Config name: react
Mode: TRAINING
Loading config from: /kaggle/working/SGG-Benchmark/configs/hydra/Spatial/react.yaml
2026-07-12 19:29:56,391 sgg_benchmark INFO: Using 1 GPUs
2026-07-12 19:29:56,391 sgg_benchmark INFO: Task mode: sgdet
2026-07-12 19:29:56,391 sgg_benchmark INFO: Loading training dataset to extract class information...
2026-07-12 19:29:56,703 sgg_benchmark INFO: Extracted from dataset: num_obj_classes=7, num_rel_classes=8
2026-07-12 19:29:56,704 sgg_benchmark INFO: Saving config to: ./checkpoints/spatial/react_auto/config.yml
2026-07-12 19:29:56,733 sgg_benchmark INFO: Saving Hydra config to: ./checkpoints/spatial/react_auto/hydra_config.yaml
2026-07-12 19:29:56,733 sgg_benchmark INFO: Building model...
Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ult

/kaggle/working/SGG-Benchmark/sgg_benchmark/engine/trainer.py:334: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"{_ALIASES.get(k, k)}={float(v):.3f}" for k, v in loss_dict_reduced.items()
SGG Eval: 100%|██████████| 100/100 [00:01<00:00, 64.52it/s]


In [10]:
%%bash
cd /kaggle/working/SGG-Benchmark
zip -rq /kaggle/working/results.zip checkpoints/spatial -x "*.pth" -x "*.pt"
echo "RESULTS -> /kaggle/working/results.zip"; ls -la /kaggle/working/results.zip

RESULTS -> /kaggle/working/results.zip
-rw-r--r-- 1 root root 63696 Jul 12 19:56 /kaggle/working/results.zip
